# pycograd — visualizing the graph IR

pycograd's **graph mode** records the primitives a `|>` pipeline runs into a flat
SSA graph (`capture`), differentiates it (`grad_graph` / `vjp_graph`), and optimizes
it (`optimize`). This notebook shows how to **see** those graphs:

* a bare `Graph` in a cell **auto-renders** (an SVG picture when `graphviz` is
  installed, otherwise a jaxpr-style text listing),
* `graph.pretty()` is the text listing, `graph.to_dot()` is Graphviz DOT source.

> For the SVG pictures you need the `graphviz` **Python package** *and* the Graphviz
> `dot` **binary** on your PATH (`pip install graphviz`, plus e.g. `brew install
> graphviz` / `apt-get install graphviz`). Without them, displaying a graph falls
> back to the text listing — everything below still works.

In [ ]:
%load_ext pycograd

## 1. Capture a `|>` pipeline into the graph IR

Parameters go in a `params{...} as weights:` block; the forward is a `|>` pipeline.
`capture(forward, x)` runs it once with a tracer input and records every op into a
`Graph`. Displaying the bare `g` auto-renders.

> `$` is the piped value for a stage. Reusing the *same* value within a stage needs a
> **named** placeholder — `$h * $h` squares `h`; a bare `$ * $` would be two holes.

In [ ]:
import numpy as np
from pycograd import capture, grad_graph, optimize, d_sigmoid, ops
from pycograd.transpose import vjp_graph

rng = np.random.default_rng(0)
x = rng.standard_normal((4, 3))

with params{
    w = rng.standard_normal((3, 2))
    b = np.zeros(2)
} as weights:
    # sum(tanh(x @ w + b) ** 2) as a pipeline; $h * $h reuses the piped value.
    forward = $ |> $ @ w + b |> np.tanh($) |> $h * $h |> np.sum($)
    g = capture(forward, x)

g            # bare Graph -> auto-renders (SVG if graphviz is installed, else the listing)

`repr` stays a terse one-liner (so logging isn't noisy); `print` / `.pretty()` give
the full SSA listing — `%id = prim args {params} -> dtype[shape]`.

In [ ]:
print(repr(g))
print()
print(g.pretty())

### The DOT source
`graph.to_dot()` returns Graphviz DOT text you can render anywhere (e.g. write to a
file and run `dot -Tpng g.dot -o g.png`). `graph.render()` returns a
`graphviz.Source` for inline display when the package is installed.

In [ ]:
print(g.to_dot())

## 2. Forward **and** backward in one graph: `grad_graph`

`grad_graph` differentiates the captured forward into a single graph computing
`(value, grads)` — forward and backward together. That's what lets the optimizer
work *across* the forward/backward boundary.

In [ ]:
gg = grad_graph(g)   # outputs: value, then one grad per input leaf
gg

## 3. Cross-pass optimization you can *see*

`d_sigmoid`'s gradient is `g · σ(x) · (1 − σ(x))` — it **recomputes** `σ(x)`. So the
combined forward+backward graph contains the forward's `σ(x)` plus the two in the
backward. `optimize` runs CSE **across the boundary** and merges them into one.

In [ ]:
sig = $ |> d_sigmoid($) |> np.sum($)    # the VJP recomputes sigmoid(x)
xs = rng.standard_normal((3, 4))
combined = grad_graph(capture(sig, xs))
opt = optimize(combined)

n_before = sum(1 for n in combined.nodes if n.prim is ops.d_sigmoid)
n_after  = sum(1 for n in opt.nodes      if n.prim is ops.d_sigmoid)
print(f"d_sigmoid nodes  before optimize: {n_before}   after: {n_after}")
opt   # the merged graph -- one sigmoid feeds both the value and the gradient

## 4. Reverse mode *derived from* forward mode: `vjp_graph`

`vjp_graph = transpose ∘ linearize` builds the same `(value, grads)` graph a
different way: it linearizes the pipeline (reusing the forward/JVP rules) into a
graph linear in the tangents, then transposes only the linear part. Same gradient,
derived from forward mode.

In [ ]:
vjp_graph(forward, x)

## 5. Watch a pass fire on the forward

`$v` (named) feeds the *same* value to both `tanh`s, so CSE can merge them; the
unused branch is dropped by DCE.

In [ ]:
redundant = $v |> np.tanh($v) * np.tanh($v) |> np.sum($)   # two tanh of the SAME input
r = capture(redundant, xs)
print("before:", repr(r))
print("after :", repr(optimize(r)))   # the duplicate tanh is gone
optimize(r)